In [ ]:
def evaluate_hypothesis_output(output_text, reference_text=None):
    results = {}

    # 1. Correctness 
    results["correctness"] = evaluate_correctness_llm(output_text)

    # 2. LLM-as-a-judge
    results["llm_judge"] = llm_judge_score(output_text, reference_text)

    # 3. Text metrics 
    if reference_text:
        results["rouge2"] = compute_rouge2(output_text, reference_text)
        results["bleu"] = compute_bleu(output_text, reference_text)

    # 4. Clinical metrics
    results["clinical_precision"] = evaluate_medical_precision(output_text)

    return results

In [ ]:
def evaluate_correctness_llm(text):
    prompt = f"""
    Is the following medical statement factually correct?
    Answer YES or NO and explain briefly.

    {text}
    """
    return call_llm(prompt)

In [ ]:
def llm_judge_score(prediction, reference):
    prompt = f"""
    You are a clinical expert.

    Compare the following:

    Prediction:
    {prediction}

    Reference (gold standard):
    {reference}

    Score from 0 to 1 based on:
    - clinical correctness
    - reasoning quality
    - safety

    Output JSON with score and explanation.
    """
    return call_llm(prompt)

In [ ]:
from collections import Counter

def rouge2_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens = ref.lower().split()

    pred_bigrams = list(zip(pred_tokens, pred_tokens[1:]))
    ref_bigrams = list(zip(ref_tokens, ref_tokens[1:]))

    pred_counts = Counter(pred_bigrams)
    ref_counts = Counter(ref_bigrams)

    overlap = sum((pred_counts & ref_counts).values())

    if len(pred_bigrams) == 0 or len(ref_bigrams) == 0:
        return 0.0

    precision = overlap / len(pred_bigrams)
    recall = overlap / len(ref_bigrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

#Question 5
ref = "Overall, high glucose and retinal vascular occlusion may be associated with diabetes-related vascular damage, but they are not reliable independent indicators of Type 2 Diabetes specifically, as these features reflect diabetes more broadly and may vary by the type of occlusion involved"

pred = full_summary  

score = rouge2_f1(pred, ref)
print("ROUGE-2:", score)

In [ ]:
def evaluate_medical_precision(text):
    keywords = ["glucose", "insulin", "diabetes", "HbA1c"]
    hits = sum(1 for k in keywords if k.lower() in text.lower())
    return hits / len(keywords

In [ ]:
from anthropic import Anthropic

client = Anthropic()

def call_llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=500,
        temperature=0,
        system="You are a clinical reasoning expert.",
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return "".join(
        block.text for block in response.content
        if getattr(block, "type", None) == "text"
    ).strip()

In [ ]:
n = min(
    len(results),
    len(PartA_conclusion_results),
    len(all_final_outputs),
    len(pipeline_results)
)

for i in range(n):
    r = results[i]

    hypothesis_text = clean_hypothesis_text(r)
    part_a_text = r["report"]
    part_a_summary = PartA_conclusion_results[i]["conclusion"]
    part_b_text = all_final_outputs[i]["final_output"]
    part_c_text = pipeline_results[i]["result"]

    full_summary = build_full_hypothesis_summary(
        hypothesis_text=hypothesis_text,
        part_a_text=part_a_text,
        part_a_summary=part_a_summary,
        part_b_text=part_b_text,
        part_c_text=part_c_text
    )

    eval_results = evaluate_hypothesis_output(
        output_text=full_summary,
        reference_text=None
    )

    print(full_summary)
    print("\n📊 Evaluation:", eval_results)

In [ ]:
import pandas as pd

rows = []

n = min(
    len(results),
    len(PartA_conclusion_results),
    len(all_final_outputs),
    len(pipeline_results)
)

for i in range(n):
    r = results[i]

    hypothesis_text = clean_hypothesis_text(r)

    part_a_text = r["report"]
    part_a_summary = PartA_conclusion_results[i]["conclusion"]
    part_b_text = all_final_outputs[i]["final_output"]
    part_c_text = pipeline_results[i]["result"]

    full_summary = build_full_hypothesis_summary(
        hypothesis_text=hypothesis_text,
        part_a_text=part_a_text,
        part_a_summary=part_a_summary,
        part_b_text=part_b_text,
        part_c_text=part_c_text
    )

    eval_results = evaluate_hypothesis_output(
        output_text=full_summary,
        reference_text=None
    )

    rows.append({
        "hypothesis": hypothesis_text,
        "output": full_summary,
        **eval_results
    })

df_eval = pd.DataFrame(rows)
df_eval.to_csv("evaluation_results.csv", index=False)

print("✅ Saved to evaluation_results.csv")